In [0]:
# ============================================================
# Section F – Delta Lake Operations  (notebook: freshkart/07_delta_ops)
# NOTE: F1 & F2 must be executed AFTER Run 2 in Section H.
# ============================================================

# ----------------------------------------------------------
# F1 – Time travel (2 marks)
# Compare selling_price for P004, P021, P050 before and after Batch 2
# ----------------------------------------------------------
_hist = (spark.sql("DESCRIBE HISTORY fk_ce_2026_07_1382.silver.products")
        .orderBy("version", ascending=False)
        .collect())
_ver_current   = _hist[0].version          # latest version (after batch 2)
_ver_before_b2 = _hist[1].version          # version just before batch 2

spark.sql(f"""
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.ops.time_travel_check AS
SELECT
    cur.product_id                                AS product_id,
    CAST(prev.selling_price AS DECIMAL(10,2))     AS selling_price_before_batch2,
    CAST(cur.selling_price  AS DECIMAL(10,2))     AS selling_price_current
FROM fk_ce_2026_07_1382.silver.products AS cur
LEFT JOIN fk_ce_2026_07_1382.silver.products VERSION AS OF {_ver_before_b2} AS prev
    ON cur.product_id = prev.product_id
WHERE cur.product_id IN ('P004', 'P021', 'P050')
""")

# ----------------------------------------------------------
# F2 – Change Data Feed (2 marks)
# Every change-feed row produced by the Batch 2 product load.
# (CDF must have been enabled on silver.products before Run 2.)
# ----------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.ops.product_changes_batch2 AS
SELECT
    CAST(product_id       AS STRING)        AS product_id,
    CAST(_change_type     AS STRING)        AS change_type,
    CAST(mrp              AS DECIMAL(10,2)) AS mrp,
    CAST(selling_price    AS DECIMAL(10,2)) AS selling_price,
    CAST(is_active        AS BOOLEAN)       AS is_active,
    CAST(_commit_version  AS BIGINT)        AS commit_version
FROM table_changes('fk_ce_2026_07_1382.silver.products', {_ver_before_b2 + 1}, {_ver_current})
""")

# ----------------------------------------------------------
# F3 – Layout (1 mark)
# Liquid clustering on gold.fact_order_line + optimise
# ----------------------------------------------------------
spark.sql("ALTER TABLE fk_ce_2026_07_1382.gold.fact_order_line CLUSTER BY (order_date_key, store_id)")
spark.sql("OPTIMIZE fk_ce_2026_07_1382.gold.fact_order_line")

# ----------------------------------------------------------
# F4 – Recovery drill (1 mark)
# Clone → delete → RESTORE  (history must show all three)
# ----------------------------------------------------------
# 1) Deep clone
spark.sql("""
CREATE OR REPLACE TABLE fk_ce_2026_07_1382.ops.orders_drill
DEEP CLONE fk_ce_2026_07_1382.silver.orders
""")

# 2) Delete S01 / S02 rows
spark.sql("DELETE FROM fk_ce_2026_07_1382.ops.orders_drill WHERE store_id IN ('S01', 'S02')")

# 3) Restore to pre-delete state via Delta RESTORE (not re-insert)
_drill_hist = (spark.sql("DESCRIBE HISTORY fk_ce_2026_07_1382.ops.orders_drill")
               .orderBy("version")
               .collect())
_clone_ver = _drill_hist[0].version        # version 0 – the clone
spark.sql(f"RESTORE TABLE fk_ce_2026_07_1382.ops.orders_drill TO VERSION AS OF {_clone_ver}")
